# Quantum Teleblur

Quantum Teleblur morphs one image into another through qubit rotations.

## Setup

In [ ]:
import getpass
import os

from moth_atlas import Atlas

if not os.environ.get("MOTH_API_KEY"):
    os.environ["MOTH_API_KEY"] = getpass.getpass("Atlas API key")

client = Atlas()  # reads MOTH_API_KEY
ENGINE = "telablur-v1"

In [ ]:
from IPython.display import Image, display


def show(images, width=200):
    """Display {label: image bytes}."""
    for label, data in images.items():
        print(label)
        display(Image(data, width=width))

## Try it

Point `IMAGE1` (source) and `IMAGE2` (target) at your own images, or keep the samples beside this notebook.

In [ ]:
IMAGE1, IMAGE2 = "image1.png", "image2.png"

source, target = open(IMAGE1, "rb").read(), open(IMAGE2, "rb").read()
show({"image1 (source)": source, "image2 (target)": target})

In [ ]:
params = {"strength": 0.5, "direction": "full", "size": 1024, "downscale": True}

input_files = {slot: client.assets.upload(path).asset_id for slot, path in {"image1": IMAGE1, "image2": IMAGE2}.items()}
job = client.jobs.submit(ENGINE, params=params, input_files=input_files)
client.jobs.wait(job.job_id, on_update=lambda s: print(s.status))
show({"source": source, "result": client.jobs.download(job.job_id), "target": target})

## Parameters

| name | type | default | what it does |
|---|---|---|---|
| `image1` (input file) | PNG, JPEG, WebP, TIFF or BMP | required | The source image. It is morphed toward `image2`. |
| `image2` (input file) | PNG, JPEG, WebP, TIFF or BMP | required | The target image. It is resized to match `image1` if the dimensions differ. |
| `mask` (input file) | PNG, JPEG, WebP, TIFF or BMP | optional | Per-pixel strength of the effect: black keeps `image1` unchanged, white applies the full effect. If omitted, `image1`'s alpha channel is used. |
| `strength` | float, 0 to 1 | 0.5 | Strength of the teleportation effect. |
| `direction` | `full`, `vertical` or `horizontal` | `full` | Which qubits get the blur rotation: `full` rotates all pixel qubits; `vertical` and `horizontal` limit the blur to one axis. |
| `size` | integer, 8 to 1024 | 1024 | Pixel budget per pass: a region up to `size × size` pixels is teleported in a single pass. |
| `downscale` | boolean | true | If true, oversized regions are downscaled before teleport and upscaled back; if false, they are tiled into sub-images. |
| `mask_bin_size` | number, 0 to 100 | 4 | Mask quantisation step as a percentage of 255. |
| `mask_min_region` | integer, at least 1 | 16 | Minimum connected mask region size in pixels; smaller regions are discarded. |

**Returns:** the processed image.

## How it works

Both images are encoded into one quantum state on $n+1$ qubits: $n$ qubits for the pixels and one selector qubit that marks which image each amplitude belongs to. Rotating the selector qubit blends the two images at the amplitude level.

## Experiments

Each experiment changes one parameter and keeps the others fixed, and runs one job per value.

### Strength

Strength of the teleportation effect, from `image1` unchanged at 0 to the full effect at 1.

In [ ]:
outputs = {}
for strength in (0.0, 0.25, 0.5, 0.75, 1.0):
    job = client.jobs.run(ENGINE, params={**params, "strength": strength}, files={"image1": IMAGE1, "image2": IMAGE2})
    outputs[f"strength = {strength}"] = client.jobs.download(job.job_id)
show(outputs, width=160)

## Questions

**What if the two images are different sizes?**

image2 is resized to match image1.

**Why did my run fail?**

A run still going after 18,000 s is cancelled and the job ends failed. Mismatched mask sizes and masks with no non-zero pixels are rejected.